# Demo recording: the final monitor on the 3 demo tasks

Records real runs for the replay website: **Qwen3-8B + the final Phase B adapter** (`agent-risk-adapter-final`, trained 11 Nov) as the monitor, **system v2** (the final system in the report: tag-filtered retrieval, **k = 8** policies, second-source check, prompt hints, all read from `src.monitor.case_inputs.VERSIONS["v2"]`), and the **Groq agent** `openai/gpt-oss-120b`.

- Each run is tried with the Groq agent up to `MAX_TRIES_PER_RUN` times. If Groq never proposes the task's target write (`delete_file`, `delete_repository`), the run is recorded with the **scripted** agent instead. Every log says which agent ran (`meta.agent` = `groq:openai/gpt-oss-120b` or `scripted`), plus `monitor_version` and `k`.
- Runs go one by one, with a pause before every Groq episode after the first (Groq free tier: 8,000 tokens/min). Every Groq episode goes through the cost guard (`src/service/cost_guard.py`) first.
- The demo tasks are new tasks on the seed-0 company (`lma_agent/tasks/demo_tasks.yaml`), never test or OOD records.
- Logic: `src/live/demo_record.py` (tested on CPU in `tests/test_demo_record.py`). This notebook only sets up, checks, runs and prints.

**Kaggle setup (once)**
1. Push the code to GitHub, then `bash scripts/kaggle_push_code.sh "demo record"` on your machine (creates/updates your private `<you>/agent-risk-code`).
2. New notebook → File → Import notebook → this file. Settings: **Accelerator GPU T4 x2**, **Internet on**.
3. Add Input → your `agent-risk-code` dataset and the shared `agent-risk-adapter-final` dataset.
4. Add-ons → Secrets → `GROQ_API_KEY` (attach it to this notebook). It is read into the environment, never printed.
5. Fill the two TODOs in the CONFIG cell, then Run All. About 20 min model load + a few minutes per run, plus the pauses.

**Outputs** (`/kaggle/working/demo/`): one `<run_id>.jsonl` per kept run, `attempts/` (Groq tries that did not reach the write, not replayed), `summary.json`, `summary.md`, `cost_ledger_kaggle.csv`.

In [ ]:
# ---------------- CONFIG: the only cell to edit ----------------
# TODO(Prateek): owner prefix exactly as in the dataset URL: kaggle.com/datasets/<owner>/agent-risk-adapter-final
ADAPTER_DATASET = "TODO_OWNER/agent-risk-adapter-final"
# TODO(Prateek): your code dataset from scripts/kaggle_push_code.sh: <your Kaggle username>/agent-risk-code
CODE_DATASET = "TODO_OWNER/agent-risk-code"

MONITOR_VERSION = "v2"        # the final system; k = 8 and every other setting come from VERSIONS["v2"]
USE_GROQ = True               # False = scripted agent only, no API calls at all
MAX_TRIES_PER_RUN = 3         # Groq tries per run before the scripted fallback
GROQ_EPISODE_BUDGET = 5       # Groq episodes for this whole session (CLAUDE.md Tier B: at most 5 agent tasks per session)
PAUSE_BETWEEN_GROQ_S = 65     # pause before each Groq episode after the first (free tier: 8,000 tokens/min)
RUNS = None                   # None = src.live.demo_record.PLAN (7 runs, the 3 selective ones first);
                              # or a list such as [("demo_hidden", "selective", None), ("demo_acmepay", "selective", "no")]
OUT = "/kaggle/working/demo"
EXPECTED_BASE_MODEL = "Qwen3-8B"

In [ ]:
# Find the two input datasets (no guessing: the CONFIG names them)
import glob, json, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path

def dataset_dir(spec: str) -> Path:
    if "TODO" in spec or spec.count("/") != 1:
        raise SystemExit(f"Fill in the owner prefix in the CONFIG cell: {spec!r}")
    owner, slug = spec.split("/")
    for cand in (Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug):
        if cand.is_dir():
            return cand
    subprocess.run(["find", "/kaggle/input", "-maxdepth", "3"], check=False)
    raise SystemExit(f"{spec} is not attached: Add Input -> {spec} (listing of /kaggle/input above)")

def one(root: Path, pattern: str) -> Path:
    hits = [Path(p) for p in glob.glob(f"{root}/**/{pattern}", recursive=True) if "/epoch_" not in p]
    if len(hits) != 1:
        raise SystemExit(f"expected exactly one {pattern} under {root}, found {hits}")
    return hits[0]

ADAPTER_DIR = one(dataset_dir(ADAPTER_DATASET), "adapter_config.json").parent
CODE_DIR = dataset_dir(CODE_DATASET)
print("adapter:", ADAPTER_DIR)
print("code:   ", CODE_DIR)

In [ ]:
# Code -> /tmp/repo (writable: the cost guard writes its ledger there), then the libraries
WORK = Path("/tmp/repo")
if not WORK.exists():
    unpacked = glob.glob(f"{CODE_DIR}/**/src/monitor/run.py", recursive=True)
    if unpacked:
        shutil.copytree(Path(unpacked[0]).parents[2], WORK)
    else:
        with tarfile.open(one(CODE_DIR, "code.tar.gz")) as tar:
            tar.extractall(WORK)
commit_file = glob.glob(f"{CODE_DIR}/**/code_commit.txt", recursive=True)
CODE_COMMIT = Path(commit_file[0]).read_text().strip() if commit_file else "unknown"
print("code commit:", CODE_COMMIT)
sys.path.insert(0, str(WORK))
os.chdir(WORK)

import importlib.util
def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod
base = load_module("run_base_eval", WORK / "kaggle/base_eval/run_base_eval.py")          # MODEL, MODEL_REVISION, PIP
finetune = load_module("run_finetune", WORK / "kaggle/finetune/run_finetune.py")         # PIP_PHASE_B
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *base.PIP, *finetune.PIP_PHASE_B, "openai"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)   # as the eval jobs

In [ ]:
# Groq key from Kaggle Secrets (never printed), then stop early if the cost guard would block Groq
if USE_GROQ:
    from kaggle_secrets import UserSecretsClient
    os.environ["GROQ_API_KEY"] = UserSecretsClient().get_secret("GROQ_API_KEY")
    from src.service.cost_guard import approved
    if not approved():
        raise SystemExit("The cost guard (src/service/cost_guard.py) blocks real-account calls in this session, "
                         "so no Groq call will be made. Stopped before loading the model. "
                         "USE_GROQ = False records the scripted agent only.")
    print("Groq agent on: key loaded, cost guard allows Tier B")
else:
    print("Groq agent off: every run uses the scripted agent")

In [ ]:
# Which adapter is this? Print its config so we can confirm it is the final Phase B adapter
ADAPTER_CONFIG = json.loads((ADAPTER_DIR / "adapter_config.json").read_text())
SHOWN = {k: ADAPTER_CONFIG.get(k) for k in ("base_model_name_or_path", "peft_type", "task_type", "r", "lora_alpha",
                                            "lora_dropout", "target_modules")}
print(json.dumps(SHOWN, indent=2))
for f in sorted(ADAPTER_DIR.iterdir()):
    print(f"  {f.name:40s} {f.stat().st_size / 2**20:8.1f} MiB")
stats_file = ADAPTER_DIR / "train_stats.json"
if stats_file.exists():
    stats = json.loads(stats_file.read_text())
    print("train_stats:", json.dumps({k: stats.get(k) for k in ("model", "rank", "alpha", "lr", "epochs", "max_len",
                                                                "train", "train_loss")}, indent=2))
    print("Final Phase B adapter (11 Nov, PROGRESS.md): train_loss 0.0308. Phase B v1 (30 Oct) was 0.0336.")
else:
    print("no train_stats.json next to the adapter: confirm the adapter by its dataset name and date")
if EXPECTED_BASE_MODEL not in str(SHOWN["base_model_name_or_path"]):
    raise SystemExit(f"adapter base model is {SHOWN['base_model_name_or_path']}, expected {EXPECTED_BASE_MODEL}")

In [ ]:
# The monitor system: every setting from the one place the evaluation runs also read
from src.monitor.case_inputs import VERSIONS
SYSTEM = VERSIONS[MONITOR_VERSION]
print(MONITOR_VERSION, SYSTEM, "-> k =", SYSTEM["shown"])
assert MONITOR_VERSION == "v2" and SYSTEM["shown"] == 8, "the demo must use the final system (v2, 8 policies)"

In [ ]:
# Load the monitor (Qwen3-8B + adapter, float16 over both T4s) and the retrieval embedder (as in eval)
import torch
subprocess.run(["nvidia-smi"], check=False)
from src.monitor.models import TransformersModel
from src.policies.retriever import BgeEmbedder
t0 = time.perf_counter()
model = TransformersModel(base.MODEL, revision=base.MODEL_REVISION, adapter=str(ADAPTER_DIR))
embedder = BgeEmbedder()
print(f"model + embedder loaded in {time.perf_counter() - t0:.0f}s")
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}, {free / 2**30:.1f} of {total / 2**30:.1f} GiB free")

In [ ]:
# Record the runs one by one
from datetime import datetime, timezone
from lma_agent.tasks.load_tasks import load_tasks
from src.live.demo_record import PLAN, GroqBudget, groq_agent, record_run

def log(msg):
    print(f"[{datetime.now(timezone.utc):%H:%M:%S} UTC] {msg}", flush=True)

TASKS = {t.task_id: t for t in load_tasks(WORK / "lma_agent/tasks/demo_tasks.yaml")}
OUT_DIR = Path(OUT)
OUT_DIR.mkdir(parents=True, exist_ok=True)
budget = GroqBudget(GROQ_EPISODE_BUDGET)
base_meta = {"adapter_dataset": ADAPTER_DATASET, "adapter_config": SHOWN, "code_commit": CODE_COMMIT,
             "embedder": "bge", "recorded_on": "kaggle"}
n_gpu = torch.cuda.device_count()
rows = []
for task_id, setting, confirm in (RUNS or PLAN):
    log(f"run: {task_id} | {setting} | confirm {confirm or 'task default'} | Groq episodes left {budget.left}")
    for i in range(n_gpu):
        torch.cuda.reset_peak_memory_stats(i)
    t0 = time.perf_counter()
    row = record_run(TASKS[task_id], setting, confirm, model=model, embedder=embedder, out_dir=OUT_DIR,
                     groq=groq_agent if USE_GROQ else None, budget=budget, max_tries=MAX_TRIES_PER_RUN,
                     pause_s=PAUSE_BETWEEN_GROQ_S, version=MONITOR_VERSION, base_meta=base_meta, log=log)
    row["wall_s"] = round(time.perf_counter() - t0, 1)
    row["peak_gpu_gib"] = [round(torch.cuda.max_memory_allocated(i) / 2**30, 2) for i in range(n_gpu)]
    rows.append(row)
    log(f"  agent {row['agent']} | decision {row['decision']} (expected {row['expected_decision']}) | "
        f"checks {row['checks_used']} (expected {row['expected_checks']}, always {row['checks_always']})")
    log(f"  monitor prompt tokens per pass {row['prompt_tokens']} | output tokens {row['output_tokens']} | "
        f"latency per pass {row['latency_s']} s | peak GPU {row['peak_gpu_gib']} GiB")
    (OUT_DIR / "summary.json").write_text(json.dumps(rows, indent=2))       # kept even if a later run fails
log(f"done: {len(rows)} runs, Groq episodes used {budget.used} of {budget.total}")

In [ ]:
# Results table (summary.md) and the cost ledger rows of this session
def cell(v):
    return "" if v is None else ", ".join(map(str, v)) if isinstance(v, list) else str(v)

head = ["task", "setting", "confirm", "agent", "monitor", "decision", "expected", "checks (used / expected / always)",
        "prompt tokens per pass", "latency s per pass", "peak GPU GiB", "match"]
lines = ["| " + " | ".join(head) + " |", "|" + "---|" * len(head)]
for r in rows:
    lines.append("| " + " | ".join(cell(x) for x in (
        r["task"], r["setting"], r["confirm"], r["agent"], f"{r['monitor_version']} (k={r['k']})", r["decision"],
        r["expected_decision"], f"{cell(r['checks_used'])} / {cell(r['expected_checks'])} / {cell(r['checks_always'])}",
        r["prompt_tokens"], r["latency_s"], r["peak_gpu_gib"], r["match"])) + " |")
all_tokens = [t for r in rows for t in r["prompt_tokens"] if t]
all_latency = [t for r in rows for t in r["latency_s"] if t]
notes = [f"Monitor: {base.MODEL} + {ADAPTER_DATASET}, system {MONITOR_VERSION} (k={SYSTEM['shown']}), code {CODE_COMMIT}.",
         f"Largest monitor prompt: {max(all_tokens) if all_tokens else 'n/a'} tokens; slowest pass: "
         f"{max(all_latency) if all_latency else 'n/a'} s on 2x T4 (for sizing the Modal deployment).",
         f"Groq episodes used: {budget.used} of {budget.total}. Fallbacks: " +
         ("; ".join(f"{r['task']} {r['setting']}: {r['fallback_reason']}" for r in rows if r.get("fallback_reason")) or "none")]
table = "\n".join(lines + [""] + notes)
(OUT_DIR / "summary.md").write_text(table + "\n")
print(table)

ledger = WORK / "ops" / "cost_ledger.csv"
if ledger.exists():
    shutil.copyfile(ledger, OUT_DIR / "cost_ledger_kaggle.csv")
    print("\ncost ledger rows written on Kaggle (add them to your local ops/cost_ledger.csv):")
    print(ledger.read_text())
else:
    print("\nno cost ledger rows (no Groq episode was started)")

**After the run**
1. Output tab → download `demo/` and put it in `outputs/live/demo/` on your machine (`outputs/live/` is git-ignored).
2. Add the rows of `cost_ledger_kaggle.csv` to your local `ops/cost_ledger.csv`.
3. Check `summary.md`: the `agent` column says which runs are Groq and which fell back to scripted. The replay site shows the same (`meta.agent`, `monitor_version`, `k`).